In [0]:
# Read the CSV file from the Unity Catalog volume
file_path = "/Volumes/workspace/default/cc_a_b_test/cookie_cats.csv"
df = spark.read.csv(file_path, header=True, inferSchema=True)

# Display the data
display(df)

In [0]:
# ---------------------------------------------------------------------------
# A/B Test Design — Cookie Cats (gate_30 vs gate_40)
# ---------------------------------------------------------------------------

# Hypotheses (two-sided test)
#   H0 (null):       7-day retention is the same for gate_30 and gate_40
#                    (p_gate30  ==  p_gate40)
#   H1 (alternative): 7-day retention differs between gate_30 and gate_40
#                    (p_gate30  !=  p_gate40)

# Metric
#   Primary metric: 7-day retention rate
#   Defined as the proportion of players who returned to the game on
#   day 7 after install  (column: retention_7, values True/False).
#   This is a binary per-user metric, so the appropriate test is a
#   two-proportion z-test (or equivalently a chi-square test of independence).

# Significance level
alpha = 0.05        # 5 % — probability of a false positive (Type I error)

# Minimum Detectable Effect (MDE)
#   The smallest absolute change in 7-day retention we want to reliably detect.
mde = 0.01           # 1 percentage point (e.g. 18 % -> 19 %)

# Power (complementary to Type II error)
#   Standard convention; probability of correctly rejecting H0 when H1 is true.
power = 0.80

print("""
A/B Test Summary — Cookie Cats
-----------------------------------------------
Hypothesis      : H0: p_gate30 == p_gate40
                  H1: p_gate30 != p_gate40
Metric          : 7-day retention rate (retention_7)
Alpha (a)       : 0.05
MDE             : 0.01 (1 percentage point)
Power (1-B)     : 0.80
Test type       : Two-sided, two-proportion z-test
-----------------------------------------------
""")


In [0]:
%pip install statsmodels

In [0]:
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize
from pyspark.sql import functions as F

# Baseline retention rate for gate_30 (observed from the data)
p_baseline = df.filter("version = 'gate_30'").select(
    (F.sum(F.col("retention_7").cast("int")) / F.count("*")).alias("rate")
).first()["rate"]

# Minimum Detectable Effect (absolute change in retention rate)
mde = 0.01

# Effect size for two-proportion test (Cohen's h)
es = proportion_effectsize(p_baseline, p_baseline - mde)

# Required sample size per group (two-sided test)
n_required = NormalIndPower().solve_power(
    effect_size=abs(es), alpha=0.05, power=0.80
)

# Actual sample sizes per group
actual_counts = (
    df.groupBy("version")
      .agg(F.count("*").alias("n"))
      .orderBy("version")
      .collect()
)
actual_dict = {row["version"]: row["n"] for row in actual_counts}

print("=" * 55)
print("Power Analysis — Sample Size per Group")
print("=" * 55)
print(f"Baseline retention (gate_30): {p_baseline:.4f}")
print(f"MDE:                        {mde:.4f}  (1 pp)")
print(f"Effect size (Cohen's h):    {abs(es):.4f}")
print(f"Alpha:                      {0.05}")
print(f"Power:                      {0.80}")
print(f"Required n per group:      {int(round(n_required)):,}")
print("-" * 55)
for v in sorted(actual_dict):
    print(f"Actual n  ({v}):          {actual_dict[v]:,}")
min_actual = min(actual_dict.values())
ratio = min_actual / n_required
print("-" * 55)
print(f"Smallest actual group:      {min_actual:,}")
print(f"Required per group:         {int(round(n_required)):,}")
if min_actual >= n_required:
    print(f"Ratio (actual / required):  {ratio:.1f}x  -> well powered")
else:
    print(f"Ratio (actual / required):  {ratio:.1f}x  -> underpowered")
print("=" * 55)

In [0]:
from pyspark.sql import functions as F
from scipy.stats import chisquare

# ---------------------------------------------------------------------------
# Sample Ratio Mismatch (SRM) check
# ---------------------------------------------------------------------------
# A 50/50 split should give roughly equal counts in each group.  If the
# observed split deviates significantly from the expected 50/50, a Sample
# Ratio Mismatch has occurred and the experiment data is unreliable.
#
# We use a one-sample chi-square goodness-of-fit test:
#   H0: observed counts follow the expected 50/50 proportions
#   H1: observed counts do not follow the expected 50/50 proportions
# ---------------------------------------------------------------------------

alpha = 0.05

# Observed user counts per variant
observed_rows = (
    df.groupBy("version")
      .agg(F.count("*").alias("n"))
      .orderBy("version")
      .collect()
)
observed = [row["n"] for row in observed_rows]
labels    = [row["version"] for row in observed_rows]
total     = sum(observed)

# Expected counts under a perfect 50/50 split
expected = [total / 2, total / 2]

# Chi-square goodness-of-fit test
chi2_stat, p_value = chisquare(f_obs=observed, f_exp=expected)

print("=" * 55)
print("Sample Ratio Mismatch (SRM) Check")
print("=" * 55)
for label, obs, exp in zip(labels, observed, expected):
    print(f"  {label}: observed = {obs:,}  expected = {exp:,.1f}  ({obs/total*100:.2f}%)")
print("-" * 55)
print(f"Chi-square statistic : {chi2_stat:.4f}")
print(f"p-value             : {p_value:.6f}")
print(f"Alpha               : {alpha}")
print("-" * 55)
if p_value < alpha:
    print("Result: p < alpha  ->  SRM DETECTED")
    print("The split is NOT 50/50; investigate before trusting experiment results.")
else:
    print("Result: p >= alpha  ->  No SRM detected")
    print("The split is consistent with a 50/50 allocation.")
print("=" * 55)

In [0]:
import numpy as np
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep

# ---------------------------------------------------------------------------
# Two-proportion z-test for 7-day retention (gate_30 vs gate_40)
# ---------------------------------------------------------------------------
#  H0:  p_gate30 == p_gate40
#  H1:  p_gate30 != p_gate40
#  Test: two-sided two-proportion z-test (pooled variance)
# ---------------------------------------------------------------------------

alpha = 0.05

# Extract success counts and sample sizes per group
agg = (
    df.groupBy("version")
      .agg(
          F.sum(F.col("retention_7").cast("int")).alias("successes"),
          F.count("*").alias("n")
      )
      .orderBy("version")
      .collect()
)

row_g30 = [r for r in agg if r["version"] == "gate_30"][0]
row_g40 = [r for r in agg if r["version"] == "gate_40"][0]

counts = np.array([row_g30["successes"], row_g40["successes"]])
nobs   = np.array([row_g30["n"], row_g40["n"]])

p_g30 = counts[0] / nobs[0]
p_g40 = counts[1] / nobs[1]
diff  = p_g30 - p_g40          # gate_30 minus gate_40

# --- Two-proportion z-test (pooled) ---
z_stat, p_value = proportions_ztest(counts, nobs, alternative="two-sided")

# --- 95% confidence interval for the difference in proportions ---
# Uses the Newcombe (score) method via statsmodels
low_ci, high_ci = confint_proportions_2indep(
    count1=counts[0], nobs1=nobs[0],
    count2=counts[1], nobs2=nobs[1],
    compare="diff", alpha=alpha, method="score"
)

# ---------------------------------------------------------------------------
# Bootstrap check (percentile CI for the difference in proportions)
# ---------------------------------------------------------------------------
np.random.seed(42)
n_boot = 10_000

# Pull the boolean arrays into pandas for fast resampling
pandas_df = df.select("version", "retention_7").toPandas()
g30 = pandas_df.loc[pandas_df["version"] == "gate_30", "retention_7"].astype(int).values
g40 = pandas_df.loc[pandas_df["version"] == "gate_40", "retention_7"].astype(int).values

boot_diffs = np.empty(n_boot)
for i in range(n_boot):
    b30 = np.random.choice(g30, size=len(g30), replace=True)
    b40 = np.random.choice(g40, size=len(g40), replace=True)
    boot_diffs[i] = b30.mean() - b40.mean()

boot_ci = np.percentile(boot_diffs, [2.5, 97.5])

# ---------------------------------------------------------------------------
# Report
# ---------------------------------------------------------------------------
print("=" * 60)
print("Two-Proportion Z-Test — 7-day Retention (gate_30 vs gate_40)")
print("=" * 60)
print(f"gate_30 : n = {nobs[0]:,}  |  retained = {counts[0]:,}  |  rate = {p_g30:.4f}")
print(f"gate_40 : n = {nobs[1]:,}  |  retained = {counts[1]:,}  |  rate = {p_g40:.4f}")
print("-" * 60)
print(f"Observed difference (gate_30 - gate_40):  {diff:.4f}  ({diff*100:.2f} pp)")
print(f"z-statistic :  {z_stat:.4f}")
print(f"p-value     :  {p_value:.6f}")
print(f"Alpha       :  {alpha}")
print("-" * 60)
print("95% Confidence Interval for the difference (Newcombe score):")
print(f"  [{low_ci:.4f},  {high_ci:.4f}]   ({low_ci*100:.2f} pp,  {high_ci*100:.2f} pp)")
print("-" * 60)
print("Bootstrap check (10 000 resamples, percentile CI):")
print(f"  Mean boot difference :  {boot_diffs.mean():.4f}")
print(f"  95% bootstrap CI     :  [{boot_ci[0]:.4f},  {boot_ci[1]:.4f}]")
print("-" * 60)
if p_value < alpha:
    print("Decision: p < alpha  ->  REJECT H0")
    print("There IS a statistically significant difference in 7-day retention.")
else:
    print("Decision: p >= alpha  ->  FAIL TO REJECT H0")
    print("There is NO statistically significant difference in 7-day retention.")
print("=" * 60)

In [0]:
# ---------------------------------------------------------------------------
# Business Recommendation — Cookie Cats A/B Test (gate_30 vs gate_40)
# ---------------------------------------------------------------------------
#  Decision framework:
#    1. Statistical significance  — did the test detect a real difference?
#    2. Practical significance   — is the effect large enough to matter?
#    3. Risk & downside          — what do we lose by switching (or not)?
#    4. Recommendation           — clear, actionable guidance for the team.
# ---------------------------------------------------------------------------

# Pull in results computed in earlier cells
#   p_value, z_stat, alpha, diff, p_g30, p_g40, low_ci, high_ci, boot_ci

significant = p_value < alpha
absolute_diff_pp = abs(diff) * 100                     # percentage points
relative_lift = abs(diff) / p_g30 * 100                # % relative change
mde_pp = mde * 100                                     # MDE in percentage points
ci_crosses_zero = (low_ci < 0 < high_ci) or (high_ci > 0 > low_ci)

# ---------------------------------------------------------------------------
# 1 — Statistical significance
# ---------------------------------------------------------------------------
print("=" * 65)
print("Business Recommendation — Cookie Cats gate_30 vs gate_40")
print("=" * 65)
print()
print("1. STATISTICAL SIGNIFICANCE")
print("-" * 65)
print(f"   p-value               : {p_value:.6f}")
print(f"   alpha                 : {alpha}")
print(f"   z-statistic           : {z_stat:.4f}")
if significant:
    print("   Result                : STATISTICALLY SIGNIFICANT (p < alpha)")
else:
    print("   Result                : NOT statistically significant (p >= alpha)")
print()

# ---------------------------------------------------------------------------
# 2 — Practical significance (effect size vs MDE)
# ---------------------------------------------------------------------------
print("2. PRACTICAL SIGNIFICANCE")
print("-" * 65)
print(f"   gate_30 retention     : {p_g30*100:.2f}%")
print(f"   gate_40 retention     : {p_g40*100:.2f}%")
print(f"   Absolute difference   : {absolute_diff_pp:.2f} pp")
print(f"   Relative change       : {relative_lift:.2f}%")
print(f"   MDE we designed for   : {mde_pp:.2f} pp")
print(f"   95% CI (difference)   : [{low_ci*100:.2f} pp, {high_ci*100:.2f} pp]")
print(f"   95% bootstrap CI      : [{boot_ci[0]*100:.2f} pp, {boot_ci[1]*100:.2f} pp]")
if absolute_diff_pp >= mde_pp:
    print(f"   Effect >= MDE         : YES — meets the pre-registered threshold")
else:
    print(f"   Effect < MDE          : Effect is below the threshold we cared about")
print()

# ---------------------------------------------------------------------------
# 3 — Risk assessment
# ---------------------------------------------------------------------------
print("3. RISK ASSESSMENT")
print("-" * 65)
if significant and diff > 0:
    direction = "gate_30 has HIGHER retention"
    print(f"   Direction             : {direction}")
    print(f"   Moving to gate_40 would lose ~{absolute_diff_pp:.2f} pp of 7-day retention.")
    print("   This translates to measurably fewer returning players and lower")
    print("   long-term engagement / monetisation potential.")
elif significant and diff < 0:
    direction = "gate_40 has HIGHER retention"
    print(f"   Direction             : {direction}")
    print(f"   Moving to gate_40 would gain ~{absolute_diff_pp:.2f} pp of 7-day retention.")
    print("   The later gate appears to keep more players engaged through day 7.")
else:
    direction = "NO significant difference"
    print(f"   Direction             : {direction}")
    print(f"   The observed difference ({absolute_diff_pp:.2f} pp) is not statistically")
    print("   significant, and the confidence interval cannot rule out a zero effect.")
    print("   Risk of switching to gate_40: low in terms of retention, but the")
    print("   change is also not backed by evidence of improvement.")
print()

# ---------------------------------------------------------------------------
# 4 — Final recommendation
# ---------------------------------------------------------------------------
print("4. RECOMMENDATION")
print("-" * 65)
if significant and diff > 0:
    print("   >>> KEEP gate_30 (current gate at level 30).")
    print("")
    print("   The data shows a statistically significant DROP in 7-day retention")
    print(f"   when moving the gate to level 40 ({absolute_diff_pp:.2f} pp, p = {p_value:.4f}).")
    print("   Because retention is a leading indicator of player lifetime value,")
    print("   we recommend NOT rolling out gate_40.  Retain the existing gate at 30.")
elif significant and diff < 0:
    print("   >>> ROLL OUT gate_40.")
    print("")
    print("   The data shows a statistically significant INCREASE in 7-day retention")
    print(f"   when moving the gate to level 40 ({absolute_diff_pp:.2f} pp, p = {p_value:.4f}).")
    print("   We recommend shipping gate_40 to all players.")
else:
    print("   >>> DO NOT SHIP gate_40 based on this test.")
    print("")
    print("   There is no statistically significant difference in 7-day retention")
    print(f"   (p = {p_value:.4f} >= {alpha}).  The confidence interval includes zero,")
    print("   so we cannot rule out that the two gates perform identically.")
    print("")
    print("   Options:")
    print("     a) Keep gate_30  — safest; the change is not justified by the data.")
    print("     b) Run a follow-up test  — if the business wants gate_40 for other")
    print("        reasons (e.g. monetisation, session length), design a new test")
    print("        with additional metrics and a larger MDE to detect smaller effects.")

print()
print("   Next steps:")
print("     - Review secondary metrics (retention_1, session count, revenue)")
print("       before making a final product decision.")
print("     - If the business is risk-tolerant and gate_40 has other benefits,")
print("       consider a larger follow-up experiment to confirm.")
print("=" * 65)